## 1. Environment & Configuration

Set up the discovery retrieval evaluation environment. Define the project paths, golden dataset location, existing Chroma vector database path, experiment configuration, and results directory.

This step only prepares the evaluation environment. It does not modify or re-index the existing vector database.

In [2]:

from pathlib import Path
from datetime import datetime
import json
import pandas as pd
import numpy as np

# Project directories
PROJECT_DIR = Path(r"G:\SEVA_Ai\SEVA_Ai")

GOLDEN_PATH = (
    PROJECT_DIR
    / "notebooks/golden_dataset/discovery/seva_discovery_golden_v1.jsonl"
)

DISCOVERY_DB_PATH = Path(r"G:\SEVA_Ai\SEVA_Ai\output\SEVA_CHILDREN\vectorstores\discovery")

RESULTS_DIR = (
    PROJECT_DIR
    / "notebooks/experiments/discovery_evaluation"
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Experiment configuration
EXPERIMENT_ID = "DISC_EVAL_001"

TOP_K_VALUES = [5, 10, 20]

RETRIEVAL_CONFIG = {
    "search_type": "similarity",
    "top_k": 20,
}

print("Experiment:", EXPERIMENT_ID)
print("Golden dataset exists:", GOLDEN_PATH.exists())
print("Discovery DB exists:", DISCOVERY_DB_PATH.exists())
print("Results directory:", RESULTS_DIR)


Experiment: DISC_EVAL_001
Golden dataset exists: True
Discovery DB exists: True
Results directory: G:\SEVA_Ai\SEVA_Ai\notebooks\experiments\discovery_evaluation


## 2. Load Golden Dataset

Load the discovery golden dataset and validate its structure before running retrieval experiments.

Check the total number of questions, unique schemes, required fields, missing values, and duplicate questions. This ensures the evaluation dataset is ready for consistent retrieval testing.

In [ ]:
# Load golden dataset
golden_df = pd.read_json(GOLDEN_PATH, lines=True)

# Validate required columns
required_columns = [
    "test_id",
    "scheme_id",
    "scheme_name",
    "question",
    "relevant_scheme_ids",
    "categories",
    "source",
    "label_status",
]

missing_columns = set(required_columns) - set(golden_df.columns)
assert not missing_columns, f"Missing columns: {missing_columns}"

# Dataset statistics
print("Total questions:", len(golden_df))
print("Unique schemes:", golden_df["scheme_id"].nunique())
print("Unique questions:", golden_df["question"].nunique())
print("Duplicate questions:", golden_df["question"].duplicated().sum())
print("Missing values:\n", golden_df[required_columns].isnull().sum())

# Inspect sample records
display(golden_df.head(5))

Total questions: 605
Unique schemes: 121
Unique questions: 605
Duplicate questions: 0
Missing values:
 test_id                0
scheme_id              0
scheme_name            0
question               0
relevant_scheme_ids    0
categories             0
source                 0
label_status           0
dtype: int64


,test_id,scheme_id,scheme_name,question,relevant_scheme_ids,categories,source,label_status
0,DISC-0001,3he_F5gB5uF80_MGg5Hz,Transport Loan Scheme,I need a loan to purchase a light motor vehicl...,[3he_F5gB5uF80_MGg5Hz],"[Transport & Infrastructure, Business & Entrep...",groq_generated,provisional_llm_generated
1,DISC-0002,3he_F5gB5uF80_MGg5Hz,Transport Loan Scheme,Looking for financing to buy a small van for g...,[3he_F5gB5uF80_MGg5Hz],"[Transport & Infrastructure, Business & Entrep...",groq_generated,provisional_llm_generated
2,DISC-0003,3he_F5gB5uF80_MGg5Hz,Transport Loan Scheme,Seeking funds to acquire an auto‑rickshaw to s...,[3he_F5gB5uF80_MGg5Hz],"[Transport & Infrastructure, Business & Entrep...",groq_generated,provisional_llm_generated
3,DISC-0004,3he_F5gB5uF80_MGg5Hz,Transport Loan Scheme,Want a term loan for a motorbike to expand my ...,[3he_F5gB5uF80_MGg5Hz],"[Transport & Infrastructure, Business & Entrep...",groq_generated,provisional_llm_generated
4,DISC-0005,3he_F5gB5uF80_MGg5Hz,Transport Loan Scheme,Need financial support to buy a light commerci...,[3he_F5gB5uF80_MGg5Hz],"[Transport & Infrastructure, Business & Entrep...",groq_generated,provisional_llm_generated


## 01. Verify Local Embedding Model

### Why are we implementing this?

We have downloaded all-MiniLM-L6-v2 from Hugging Face.

Before loading it, we need to verify that the model weights, tokenizer and configuration files exist in our local directory.

This ensures that we can load the model without downloading files again.

In [5]:
from pathlib import Path

MODEL_PATH = Path(
    r"G:\SEVA_Ai\SEVA_Ai\models\all-MiniLM-L6-v2"
)

required_files = [
    "config.json",
    "model.safetensors",
    "modules.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "vocab.txt",
    "1_Pooling/config.json"
]

missing_files = [
    file for file in required_files
    if not (MODEL_PATH / file).exists()
]

if not missing_files:
    print("Model files verified successfully!")
    print("Model location:", MODEL_PATH)
else:
    print("Missing files:", missing_files)

Model files verified successfully!
Model location: G:\SEVA_Ai\SEVA_Ai\models\all-MiniLM-L6-v2


In [6]:
import os
import time

os.environ["HF_HUB_OFFLINE"] = "1"

from langchain_huggingface import HuggingFaceEmbeddings

start_time = time.perf_counter()

embedding_model = HuggingFaceEmbeddings(
    model_name=str(MODEL_PATH),
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

load_time = time.perf_counter() - start_time

test_embedding = embedding_model.embed_query(
    "What government schemes are available for farmers?"
)

print(f"Model loading time: {load_time:.2f} seconds")
print("Embedding dimensions:", len(test_embedding))
print("Model loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loading time: 227.17 seconds
Embedding dimensions: 384
Model loaded successfully!


In [10]:
import json
import time
from pathlib import Path
from collections import defaultdict

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

VECTORSTORE_DIR = Path(r"G:\SEVA_Ai\SEVA_Ai\output\SEVA_CHILDREN\vectorstores")

discovery_db = Chroma(
    collection_name="seva_discovery_v1",
    embedding_function=embedding_model,
    persist_directory=str(VECTORSTORE_DIR / "discovery")
)

detail_db = Chroma(
    collection_name="seva_detail_v1",
    embedding_function=embedding_model,
    persist_directory=str(VECTORSTORE_DIR / "detail")
)

discovery_count = discovery_db._collection.count()
detail_count = detail_db._collection.count()

print("Discovery collection:", discovery_count)
print("Detail collection:", detail_count)
print("Total indexed chunks:", discovery_count + detail_count)

Discovery collection: 23438
Detail collection: 67945
Total indexed chunks: 91383


In [11]:
# Retrieval configuration
RETRIEVAL_K = 30

def retrieve_schemes(query: str, k: int = RETRIEVAL_K):
    start = time.perf_counter()

    results = discovery_db.similarity_search_with_score(
        query=query,
        k=k
    )

    latency = time.perf_counter() - start

    # Deduplicate chunks into ranked schemes
    ranked_schemes = []
    seen_scheme_ids = set()

    for rank, (doc, score) in enumerate(results, start=1):
        scheme_id = doc.metadata.get("scheme_id")

        if scheme_id and scheme_id not in seen_scheme_ids:
            seen_scheme_ids.add(scheme_id)

            ranked_schemes.append({
                "scheme_id": scheme_id,
                "rank": len(ranked_schemes) + 1,
                "chunk_rank": rank,
                "distance": float(score),
                "section": doc.metadata.get("child_type")
            })

    return {
        "query": query,
        "retrieved_chunks": len(results),
        "retrieved_schemes": ranked_schemes,
        "latency": latency
    }


# Test retrieval
test_query = golden_df.iloc[0]["question"]

test_result = retrieve_schemes(test_query)

print("Query:", test_query)
print("Retrieved chunks:", test_result["retrieved_chunks"])
print("Unique schemes:", len(test_result["retrieved_schemes"]))
print("Latency:", round(test_result["latency"], 3), "seconds")

for scheme in test_result["retrieved_schemes"][:10]:
    print(
        f"Rank: {scheme['rank']} | "
        f"Scheme ID: {scheme['scheme_id']} | "
        f"Chunk Rank: {scheme['chunk_rank']}"
    )

Query: I need a loan to purchase a light motor vehicle for my transport business.
Retrieved chunks: 30
Unique schemes: 19
Latency: 0.503 seconds
Rank: 1 | Scheme ID: ZaL-yJoBXu-iUKXqmUCT | Chunk Rank: 1
Rank: 2 | Scheme ID: GMZdU5kB0AGc0F0ifzmY | Chunk Rank: 2
Rank: 3 | Scheme ID: hqIa0pwBXu-iUKXq00o3 | Chunk Rank: 3
Rank: 4 | Scheme ID: 3he_F5gB5uF80_MGg5Hz | Chunk Rank: 6
Rank: 5 | Scheme ID: ksaOaZsB0AGc0F0ilf7I | Chunk Rank: 7
Rank: 6 | Scheme ID: dca3F5gB0AGc0F0iMjSR | Chunk Rank: 8
Rank: 7 | Scheme ID: oKLGz5oBXu-iUKXqxUAg | Chunk Rank: 10
Rank: 8 | Scheme ID: mcb3T5sB0AGc0F0iTP2L | Chunk Rank: 13
Rank: 9 | Scheme ID: 6aJOaZsBXu-iUKXqSUKJ | Chunk Rank: 14
Rank: 10 | Scheme ID: hca_2ZoB0AGc0F0iCPyl | Chunk Rank: 15


In [12]:
# Run retrieval for all golden questions
evaluation_results = []

for _, row in golden_df.iterrows():
    result = retrieve_schemes(row["question"])

    evaluation_results.append({
        "test_id": row["test_id"],
        "question": row["question"],
        "expected_scheme_id": row["scheme_id"],
        "relevant_scheme_ids": row["relevant_scheme_ids"],
        "categories": row["categories"],
        "retrieved_chunks": result["retrieved_chunks"],
        "retrieved_schemes": result["retrieved_schemes"],
        "ranked_scheme_ids": [
            item["scheme_id"]
            for item in result["retrieved_schemes"]
        ],
        "latency": result["latency"]
    })

# Convert results into DataFrame
retrieval_df = pd.DataFrame(evaluation_results)

# Save raw retrieval results
retrieval_output_path = RESULTS_DIR / "discovery_retrieval_results.jsonl"

retrieval_df.to_json(
    retrieval_output_path,
    orient="records",
    lines=True,
    force_ascii=False
)

print("Total evaluated questions:", len(retrieval_df))
print("Average retrieval latency:", round(retrieval_df["latency"].mean(), 3), "seconds")
print("Average retrieved chunks:", round(retrieval_df["retrieved_chunks"].mean(), 2))
print("Average unique schemes:", round(
    retrieval_df["ranked_scheme_ids"].apply(len).mean(), 2
))
print("Saved:", retrieval_output_path)

Total evaluated questions: 605
Average retrieval latency: 0.196 seconds
Average retrieved chunks: 30.0
Average unique schemes: 21.32
Saved: G:\SEVA_Ai\SEVA_Ai\notebooks\experiments\discovery_evaluation\discovery_retrieval_results.jsonl


## 5. Configure Ragas Evaluation

Prepare Ragas for ID-based discovery retrieval evaluation.

Use IDBasedContextPrecision and IDBasedContextRecall to compare the ranked retrieved scheme IDs against the relevant scheme IDs in the golden dataset.

These metrics do not require an LLM evaluator because relevance is determined directly from scheme IDs.

In [13]:
import ragas

from ragas import SingleTurnSample
from ragas.metrics import (
    IDBasedContextPrecision,
    IDBasedContextRecall
)

print("Ragas version:", ragas.__version__)
print("ID-based precision: Available")
print("ID-based recall: Available")

Ragas version: 0.4.4
ID-based precision: Available
ID-based recall: Available


C:\Users\devendra\AppData\Local\Temp\ipykernel_11924\1960258666.py:4: DeprecationWarning: Importing IDBasedContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0 and has no direct equivalent in 'ragas.metrics.collections'.
  from ragas.metrics import (
C:\Users\devendra\AppData\Local\Temp\ipykernel_11924\1960258666.py:4: DeprecationWarning: Importing IDBasedContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0 and has no direct equivalent in 'ragas.metrics.collections'.
  from ragas.metrics import (


## 5. Configure Ragas ID-Based Metrics

Initialize Ragas ID-based context precision and context recall metrics.

In Ragas 0.4.4, these two metrics remain available through the legacy metrics API. The collections-based API does not expose equivalent ID-based classes in this release.

These metrics compare retrieved scheme IDs against the known relevant scheme IDs without requiring an LLM judge.

In [15]:
from ragas.metrics import (
    IDBasedContextPrecision,
    IDBasedContextRecall
)

id_precision = IDBasedContextPrecision()
id_recall = IDBasedContextRecall()

print("Precision:", type(id_precision).__name__)
print("Recall:", type(id_recall).__name__)

Precision: IDBasedContextPrecision
Recall: IDBasedContextRecall


C:\Users\devendra\AppData\Local\Temp\ipykernel_11924\1407634038.py:1: DeprecationWarning: Importing IDBasedContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0 and has no direct equivalent in 'ragas.metrics.collections'.
  from ragas.metrics import (
C:\Users\devendra\AppData\Local\Temp\ipykernel_11924\1407634038.py:1: DeprecationWarning: Importing IDBasedContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0 and has no direct equivalent in 'ragas.metrics.collections'.
  from ragas.metrics import (


## 6. Run Ragas ID-Based Retrieval Evaluation

Evaluate all 605 discovery queries using Ragas IDBasedContextPrecision and IDBasedContextRecall.

For each query:
- Compare ranked unique scheme IDs against the known relevant scheme IDs.
- Calculate ID-based context precision.
- Calculate ID-based context recall.
- Store individual query scores.

Aggregate the scores to measure overall discovery retrieval performance.

This evaluation uses direct ID matching and does not require an LLM judge.

In [16]:
from ragas import SingleTurnSample

ragas_results = []

for _, row in retrieval_df.iterrows():

    sample = SingleTurnSample(
        retrieved_context_ids=row["ranked_scheme_ids"],
        reference_context_ids=row["relevant_scheme_ids"]
    )

    precision_score = await id_precision.single_turn_ascore(sample)
    recall_score = await id_recall.single_turn_ascore(sample)

    ragas_results.append({
        "test_id": row["test_id"],
        "scheme_id": row["expected_scheme_id"],
        "precision": float(precision_score),
        "recall": float(recall_score)
    })

# Create results DataFrame
ragas_df = pd.DataFrame(ragas_results)

# Aggregate metrics
ragas_summary = pd.DataFrame([{
    "total_queries": len(ragas_df),
    "context_precision": ragas_df["precision"].mean(),
    "context_recall": ragas_df["recall"].mean()
}])

display(ragas_summary.style.format({
    "context_precision": "{:.4f}",
    "context_recall": "{:.4f}"
}))

# Save results
ragas_df.to_csv(
    RESULTS_DIR / "ragas_per_query_results.csv",
    index=False
)

ragas_summary.to_csv(
    RESULTS_DIR / "ragas_summary.csv",
    index=False
)

print("Ragas evaluation completed:", len(ragas_df))

,total_queries,context_precision,context_recall
0,605,0.0254,0.4860


Ragas evaluation completed: 605


## 7. Ranking-Based Retrieval Evaluation

Evaluate the ranking quality of the discovery retriever using MRR, NDCG@K, and Recall@K.

Calculate these metrics at K = 5, 10, and 20 using ranked unique scheme IDs.

The evaluation uses the known relevant scheme IDs from the provisional golden dataset. Results should be interpreted as initial retrieval benchmarks, not exhaustive relevance judgments.

In [17]:
import math

def evaluate_ranking(ranked_ids, relevant_ids, k):
    relevant_ids = set(relevant_ids)
    top_k = ranked_ids[:k]

    relevant_ranks = [
        rank
        for rank, scheme_id in enumerate(top_k, start=1)
        if scheme_id in relevant_ids
    ]

    recall = (
        len(relevant_ranks) / len(relevant_ids)
        if relevant_ids else 0
    )

    reciprocal_rank = (
        1 / relevant_ranks[0]
        if relevant_ranks else 0
    )

    dcg = sum(
        1 / math.log2(rank + 1)
        for rank in relevant_ranks
    )

    ideal_count = min(len(relevant_ids), k)

    idcg = sum(
        1 / math.log2(rank + 1)
        for rank in range(1, ideal_count + 1)
    )

    ndcg = dcg / idcg if idcg else 0

    return recall, reciprocal_rank, ndcg


ranking_results = []

for k in [5, 10, 20]:
    scores = [
        evaluate_ranking(
            row["ranked_scheme_ids"],
            row["relevant_scheme_ids"],
            k
        )
        for _, row in retrieval_df.iterrows()
    ]

    score_df = pd.DataFrame(
        scores,
        columns=["Recall", "RR", "NDCG"]
    )

    ranking_results.append({
        "K": k,
        "Recall@K": score_df["Recall"].mean(),
        "MRR@K": score_df["RR"].mean(),
        "NDCG@K": score_df["NDCG"].mean()
    })

ranking_summary = pd.DataFrame(ranking_results)

display(ranking_summary.style.format({
    "Recall@K": "{:.4f}",
    "MRR@K": "{:.4f}",
    "NDCG@K": "{:.4f}"
}))

ranking_summary.to_csv(
    RESULTS_DIR / "ranking_metrics_summary.csv",
    index=False
)

,K,Recall@K,MRR@K,NDCG@K
0,5,0.3091,0.1983,0.2259
1,10,0.3967,0.2102,0.2544
2,20,0.4694,0.2151,0.2726


## 8. Retrieval Depth Experiment

Investigate how retrieval depth affects discovery performance.

Evaluate the following Chroma retrieval depths:

- 10 chunks
- 20 chunks
- 30 chunks
- 50 chunks
- 100 chunks

For each retrieval depth:
1. Retrieve chunks using the existing similarity search.
2. Deduplicate results using scheme_id while preserving ranking.
3. Calculate Recall@5, Recall@10, Recall@20, MRR@K and NDCG@K.
4. Measure average retrieval latency and unique schemes retrieved.
5. Save per-query results and aggregate experiment metrics.

Keep the retrieval pipeline, embedding model, golden dataset and evaluation methodology unchanged.

The objective is to identify the retrieval depth that provides the best balance between retrieval quality and latency.

In [18]:
# Experiment configuration
RETRIEVAL_DEPTHS = [10, 20, 30, 50, 100]
EVALUATION_K = [5, 10, 20]

depth_results = []
depth_summary = []

for depth in RETRIEVAL_DEPTHS:

    print(f"\nRunning retrieval depth: {depth}")

    for _, row in golden_df.iterrows():

        result = retrieve_schemes(
            query=row["question"],
            k=depth
        )

        ranked_ids = [
            item["scheme_id"]
            for item in result["retrieved_schemes"]
        ]

        record = {
            "test_id": row["test_id"],
            "retrieval_depth": depth,
            "relevant_scheme_ids": row["relevant_scheme_ids"],
            "ranked_scheme_ids": ranked_ids,
            "retrieved_chunks": result["retrieved_chunks"],
            "unique_schemes": len(ranked_ids),
            "latency": result["latency"]
        }

        depth_results.append(record)

    # Evaluate this retrieval depth
    current_results = [
        r for r in depth_results
        if r["retrieval_depth"] == depth
    ]

    summary = {
        "retrieval_depth": depth,
        "avg_chunks": np.mean([
            r["retrieved_chunks"] for r in current_results
        ]),
        "avg_unique_schemes": np.mean([
            r["unique_schemes"] for r in current_results
        ]),
        "avg_latency": np.mean([
            r["latency"] for r in current_results
        ])
    }

    for k in EVALUATION_K:

        scores = [
            evaluate_ranking(
                r["ranked_scheme_ids"],
                r["relevant_scheme_ids"],
                k
            )
            for r in current_results
        ]

        score_df = pd.DataFrame(
            scores,
            columns=["Recall", "RR", "NDCG"]
        )

        summary[f"Recall@{k}"] = score_df["Recall"].mean()
        summary[f"MRR@{k}"] = score_df["RR"].mean()
        summary[f"NDCG@{k}"] = score_df["NDCG"].mean()

    depth_summary.append(summary)

    print(
        f"Completed {depth}: "
        f"Average latency = {summary['avg_latency']:.3f}s"
    )


# Create experiment DataFrames
depth_results_df = pd.DataFrame(depth_results)
depth_summary_df = pd.DataFrame(depth_summary)

# Save experiment outputs
depth_results_df.to_json(
    RESULTS_DIR / "retrieval_depth_per_query.jsonl",
    orient="records",
    lines=True
)

depth_summary_df.to_csv(
    RESULTS_DIR / "retrieval_depth_summary.csv",
    index=False
)

# Display results
display(
    depth_summary_df.style.format({
        col: "{:.4f}"
        for col in depth_summary_df.select_dtypes(
            include="number"
        ).columns
        if col != "retrieval_depth"
    })
)

print("\nRetrieval depth experiment completed.")


Running retrieval depth: 10
Completed 10: Average latency = 0.306s

Running retrieval depth: 20
Completed 20: Average latency = 0.335s

Running retrieval depth: 30
Completed 30: Average latency = 0.327s

Running retrieval depth: 50
Completed 50: Average latency = 0.364s

Running retrieval depth: 100
Completed 100: Average latency = 0.341s


,retrieval_depth,avg_chunks,avg_unique_schemes,avg_latency,Recall@5,MRR@5,NDCG@5,Recall@10,MRR@10,NDCG@10,Recall@20,MRR@20,NDCG@20
0,10,10.0000,7.6793,0.3065,0.3058,0.1977,0.2246,0.3570,0.2050,0.2417,0.3570,0.2050,0.2417
1,20,20.0000,14.6595,0.3347,0.3091,0.1983,0.2259,0.3950,0.2100,0.2539,0.4364,0.2131,0.2647
2,30,30.0000,21.3174,0.3269,0.3091,0.1983,0.2259,0.3967,0.2102,0.2544,0.4694,0.2151,0.2726
3,50,50.0000,34.4463,0.3637,0.3091,0.1983,0.2259,0.3967,0.2102,0.2544,0.4694,0.2151,0.2726
4,100,100.0000,64.8017,0.3411,0.3091,0.1983,0.2259,0.3967,0.2102,0.2544,0.4694,0.2151,0.2726



Retrieval depth experiment completed.


Recommended retrieval depth
Baseline

30 chunks
- Average unique schemes: 21.32
- Recall@20: 46.94%
- No measured recall improvement at 50 or 100 chunks.

## 9. Discovery Retrieval Error Analysis

Identify queries where the known relevant scheme was not retrieved within the top 20 unique schemes.

Inspect:
- Citizen question
- Expected scheme ID and name
- Relevant scheme rank, if retrieved outside the top 20
- Retrieved scheme IDs
- Scheme categories

Export missed queries for manual inspection.

The objective is to identify retrieval failure patterns before implementing hybrid search or reranking.

In [ ]:
# Select results from retrieval depth 30
baseline_df = depth_results_df[
    depth_results_df["retrieval_depth"] == 30
].copy()

def find_relevant_rank(ranked_ids, relevant_ids):
    ranks = [
        rank
        for rank, scheme_id in enumerate(ranked_ids, start=1)
        if scheme_id in set(relevant_ids)
    ]
    return min(ranks) if ranks else None

baseline_df["relevant_rank"] = baseline_df.apply(
    lambda row: find_relevant_rank(
        row["ranked_scheme_ids"],
        row["relevant_scheme_ids"]
    ),
    axis=1
)

# Identify queries missing the relevant scheme from top 20
missed_df = baseline_df[
    baseline_df["relevant_rank"].isna()
    | (baseline_df["relevant_rank"] > 20)
].copy()

# Join question and scheme information
missed_df = missed_df.merge(
    golden_df[["test_id", "scheme_name", "question", "categories"]],
    on="test_id",
    how="left"
)

# Export missed queries
missed_output = RESULTS_DIR / "discovery_missed_queries.csv"

missed_df[[
    "test_id",
    "question",
    "scheme_id",
    "scheme_name",
    "categories",
    "relevant_rank",
    "ranked_scheme_ids"
]].to_csv(missed_output, index=False)

print("Total queries:", len(baseline_df))
print("Missed queries at Recall@20:", len(missed_df))
print("Miss rate:", round(len(missed_df) / len(baseline_df) * 100, 2), "%")
print("Saved:", missed_output)

display(
    missed_df[[
        "test_id",
        "question",
        "scheme_name",
        "categories",
        "relevant_rank"
    ]].head(15)
)